# iGEM NLP 4 - Group models (run after notebook 3)

**What it does.** It reads the table that notebook 3 made (`pooled_master.csv`: 1,505 answers, 332 people, 125 teams) and runs the group analyses behind the website: shares of people with 95% ranges, tests that shuffle whole teams, and forest plots (part 1, the same as the website of 8 Nov 2026). Then it adds: (2) the award ladder (none, nominated, special prize) and the medal ladder with an ordinal model; (3) one model with all factors, on people; (4) tone as a ladder; (5) the smallest difference each comparison could detect.

**How to run.** Runtime > Run all. No GPU needed. It asks once for access to Google Drive. At the end a zip of the outputs downloads.

**Nothing to change.** Every cell prints its result in plain words. Raw survey files and the team roster are not used.

## Setup: find notebook 3's table and check it

In [ ]:
# Where the files are. Nothing to change if NB3's output table is on your Google Drive.
import os, glob, hashlib, datetime, csv
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    hits = sorted(glob.glob("/content/drive/MyDrive/**/pooled_master.csv", recursive=True), key=os.path.getmtime, reverse=True)
    if hits:
        INPUT_CSV = hits[0]
    else:
        from google.colab import files
        print("pooled_master.csv was not found on Drive. Please choose it from your computer (NB3 output, 04_Tables).")
        up = files.upload(); INPUT_CSV = "/content/" + list(up)[0]
    OUT_DIR = "/content/NB4_outputs"
else:
    INPUT_CSV = os.environ["NB4_INPUT"]; OUT_DIR = os.environ["NB4_TEST_OUT"]
os.makedirs(OUT_DIR, exist_ok=True)
rows = list(csv.DictReader(open(INPUT_CSV, encoding="utf-8")))
print("Input table:", INPUT_CSV)
print("  fingerprint (sha256, first 12):", hashlib.sha256(open(INPUT_CSV, "rb").read()).hexdigest()[:12])
print("  answers:", len(rows), "| people:", len({r["Participant"] for r in rows}), "| teams:", len({r["Team"] for r in rows}))
assert len(rows) == 1505 and len({r["Participant"] for r in rows}) == 332 and len({r["Team"] for r in rows}) == 125, "This is not the NB3 table of the frozen run (1,505 / 332 / 125)."
print("OK: this is the frozen NB3 table. Run date:", datetime.date.today())

## Part 1: the website analysis (people level), reproduced

In [ ]:
"""Round-4 site numbers for the redesigned iGEM page (8 Nov 2026): every figure and every test at the PEOPLE level.

Decisions (03_DECISIONS, 8 Nov): WIDE theme view (1,312 answers with a theme); base of every % = people with at least one
answer that has a theme (328 of 332; the same base as the HAL report: 162 undergraduates, 59 high school, 122 Asia).
Unit of every interval and test = the team: 2,000 team-bootstrap resamples; 4,000 shuffles of group labels between teams;
Benjamini-Hochberg over the themes of each comparison, and over the cells of each table for the circle rings.
Sub-theme tables use the sub-themes voiced by at least 15 people (the site's claim rule).
Source: pooled_master.csv, written by notebook 3.
Writes: igem-r4.json (counts and statistics only, no text) and a short text summary.
"""
import csv, json, collections, math
import numpy as np

M = list(csv.DictReader(open(INPUT_CSV, encoding="utf-8")))
NPERM, NBOOT, NLOO = 4000, 2000, 1000
rng = np.random.default_rng(20261111)
TONE = {"strongly_negative": "neg", "negative": "neg", "mixed": "mix", "neutral": "neu", "positive": "pos", "strongly_positive": "pos"}
TSC = {"neg": -1, "mix": 0, "neu": 0, "pos": 1}
REG = {"europe": "Europe", "asia": "Asia", "north-america": "North America"}
SEC = {"high-school": "High school", "undergrad": "Undergraduate", "overgrad": "Graduate"}
CNAME = {"CHN": "China", "DEU": "Germany", "CAN": "Canada", "USA": "USA", "FRA": "France", "TWN": "Taiwan", "IND": "India",
         "NLD": "Netherlands", "HKG": "Hong Kong"}
EUROPE = {"DEU": "Germany", "FRA": "France", "NLD": "Netherlands", "FIN": "Northern Europe", "SWE": "Northern Europe",
          "DNK": "Northern Europe", "LVA": "Northern Europe", "LTU": "Northern Europe", "GRC": "Southern Europe", "ITA": "Southern Europe",
          "CZE": "Central-Eastern Europe", "AUT": "Central-Eastern Europe", "CHE": "Central-Eastern Europe", "POL": "Central-Eastern Europe",
          "ROU": "Central-Eastern Europe"}
SHORT = {"Innovation Focus": "Innovation", "Inspirational Yet Challenging Experience": "Inspiring but hard",
         "Financial Limitations and Funding Issues": "Money and funding", "Collaboration Challenges and Support": "Teamwork, good and hard",
         "Time Constraints and School Pressure": "Time and school pressure", "Strong Team Connections": "Team bonds",
         "Increased Confidence and Career Clarity": "Confidence and career"}

In [ ]:
# ---- people, teams, themes (wide) --------------------------------------------------------------------------
teams = sorted({r["Team"] for r in M}); T = len(teams); ti = {t: i for i, t in enumerate(teams)}
info = {}
for r in M:
    info[r["Team"]] = dict(region=REG.get(r["Region"], "Other regions"), country=r["Country"], section=SEC[r["Section"]],
                           medal=r["medal"], special=r["special"] == "True", nominated=r["nominated"] == "True", village=r["Track"])
people_all = sorted({r["Participant"] for r in M})
themed = sorted({r["Participant"] for r in M if r["main_id_wide"] != "-1"})      # base of every %
pi = {p: i for i, p in enumerate(themed)}
p_team = {r["Participant"]: r["Team"] for r in M}
tp = np.array([ti[p_team[p]] for p in themed])                                     # team index of each base person
MAIN = {}; SUB = {}
for r in M:
    if r["main_id_wide"] != "-1": MAIN[int(r["main_id_wide"])] = r["main_name_wide"]
    if r["sub_id_wide"] != "-1": SUB[int(r["sub_id_wide"])] = (r["sub_name_wide"], int(r["main_id_wide"]))
def vmat(field, ids):
    V = np.zeros((len(themed), len(ids)), bool); col = {t: j for j, t in enumerate(ids)}
    for r in M:
        t = int(r[field])
        if t in col: V[pi[r["Participant"]], col[t]] = True
    return V
MID = sorted(MAIN); SID_ALL = sorted(SUB)
VM = vmat("main_id_wide", MID); VSA = vmat("sub_id_wide", SID_ALL)
SID = [t for j, t in enumerate(SID_ALL) if VSA[:, j].sum() >= 15]; VS = VSA[:, [SID_ALL.index(t) for t in SID]]
# per team: n base people, k people voicing each theme
NP = np.bincount(tp, minlength=T).astype(float)
def team_k(V): return np.array([V[tp == i].sum(0) for i in range(T)], float)
KM, KS = team_k(VM), team_k(VS)
# tone per team (scored answers): counts and score sum
TN = np.zeros(T); TS = np.zeros(T); TPOS = np.zeros(T); TNEG = np.zeros(T)
for r in M:
    if r["sent_status"] == "scored":
        i = ti[r["Team"]]; c = TONE[r["sent_label"]]; TN[i] += 1; TS[i] += TSC[c]; TPOS[i] += c == "pos"; TNEG[i] += c == "neg"

In [ ]:
# ---- statistics helpers (same as round 3, applied to people) ----------------------------------------------------
def bh(p):
    p = np.asarray(p, float); o = np.argsort(p); q = np.empty_like(p); m = len(p); prev = 1.0
    for rank in range(m - 1, -1, -1):
        i = o[rank]; prev = min(prev, p[i] * m / (rank + 1)); q[i] = prev
    return q
def onehot(lab, groups):
    G = np.zeros((len(lab), len(groups)))
    for i, g in enumerate(lab):
        if g in groups: G[i, groups.index(g)] = 1
    return G
def chi2_cols(G, n, k):
    gn = G.T @ n; gk = G.T @ k; tot = gn.sum(); col = gk.sum(0)
    E1 = np.outer(gn, col) / tot; E0 = np.outer(gn, tot - col) / tot
    return (((gk - E1) ** 2) / np.maximum(E1, 1e-9) + (((gn[:, None] - gk) - E0) ** 2) / np.maximum(E0, 1e-9)).sum(0)
def adj_resid(G, n, k):
    gn = G.T @ n; gk = G.T @ k; tot = gn.sum(); col = gk.sum(0); E = np.outer(gn, col) / tot
    V = E * (1 - gn[:, None] / tot) * (1 - col[None, :] / tot)
    return (gk - E) / np.sqrt(np.maximum(V, 1e-12))
def tone_stat(G, tn, ts):
    gn = G.T @ tn; m = (G.T @ ts) / np.maximum(gn, 1); mu = ts.sum() / tn.sum()
    return float((gn * (m - mu) ** 2).sum())

def table(lab, groups, K, names, keep=None):
    idx = np.array([i for i in range(T) if lab[i] in groups and (keep is None or keep[i])])
    G = onehot([lab[i] for i in idx], groups); n, k = NP[idx], K[idx]
    obs = chi2_cols(G, n, k); res = adj_resid(G, n, k)
    tn, ts = TN[idx], TS[idx]; tobs = tone_stat(G, tn, ts)
    ge = np.zeros(K.shape[1]); ce = np.zeros(res.shape); tge = 0
    for _ in range(NPERM):
        Gp = G[rng.permutation(len(idx))]
        ge += chi2_cols(Gp, n, k) >= obs - 1e-9; ce += np.abs(adj_resid(Gp, n, k)) >= np.abs(res) - 1e-9
        tge += tone_stat(Gp, tn, ts) >= tobs - 1e-12
    p = (ge + 1) / (NPERM + 1); pc = (ce + 1) / (NPERM + 1); q = bh(p); qc = bh(pc.ravel()).reshape(pc.shape)
    gn = G.T @ n; share = (G.T @ k) / gn[:, None]
    lo = np.zeros(share.shape); hi = np.zeros(share.shape); tpos = []; tneg = []; tmean = []
    for g in range(len(groups)):
        mem = np.where(G[:, g] == 1)[0]; bs = []
        for _ in range(NBOOT):
            b = rng.choice(mem, len(mem)); bs.append(k[b].sum(0) / n[b].sum())
        lo[g], hi[g] = np.percentile(bs, [2.5, 97.5], axis=0)
        ii = idx[mem]; tpos.append(TPOS[ii].sum() / TN[ii].sum()); tneg.append(TNEG[ii].sum() / TN[ii].sum()); tmean.append(TS[ii].sum() / TN[ii].sum())
    out = dict(groups=groups, teams=[int(G[:, g].sum()) for g in range(len(groups))], people=[int(x) for x in gn],
               themes=names, share=share.round(4).tolist(), lo=lo.round(4).tolist(), hi=hi.round(4).tolist(),
               all=(k.sum(0) / n.sum()).round(4).tolist(), p=p.round(4).tolist(), q=q.round(4).tolist(),
               resid=res.round(2).tolist(), cell_q=qc.round(4).tolist(),
               tone=dict(pos=np.round(tpos, 4).tolist(), neg=np.round(tneg, 4).tolist(), mean=np.round(tmean, 3).tolist(),
                         p=round((tge + 1) / (NPERM + 1), 4)))
    # leave one team out for every theme that holds
    out["loo"] = {}
    for j in np.where(q < 0.05)[0]:
        worst = 0
        for drop in range(len(idx)):
            keepi = np.array([i for i in range(len(idx)) if i != drop]); Gk = G[keepi]; nn = n[keepi]; kk = k[keepi][:, [j]]
            o = chi2_cols(Gk, nn, kk)[0]; ge1 = 0
            for _ in range(NLOO // 4):
                ge1 += chi2_cols(Gk[rng.permutation(len(keepi))], nn, kk)[0] >= o - 1e-9
            worst = max(worst, (ge1 + 1) / (NLOO // 4 + 1))
        out["loo"][names[j]] = round(worst, 4)
    return out

def both(lab, groups, keep=None):
    return dict(main=table(lab, groups, KM, [MAIN[t] for t in MID], keep), sub=table(lab, groups, KS, [SUB[t][0] for t in SID], keep))

lab = lambda f: [info[t][f] for t in teams]
cteams = collections.Counter(info[t]["country"] for t in teams)
big = [c for c, n in cteams.most_common() if n >= 5]
lab_country = [CNAME[info[t]["country"]] if info[t]["country"] in big else "Other countries" for t in teams]
lab_europe = [EUROPE.get(info[t]["country"], "-") for t in teams]
vil = [v for v, n in collections.Counter(lab("village")).most_common() if n >= 8]
R = {}
R["country"] = both(lab_country, [CNAME[c] for c in big] + ["Other countries"])
R["region"] = both(lab("region"), ["Asia", "Europe", "North America", "Other regions"])
R["region_undergrad"] = both(lab("region"), ["Asia", "Europe", "North America"], keep=[info[t]["section"] == "Undergraduate" for t in teams])
R["europe"] = both(lab_europe, ["Germany", "France", "Netherlands", "Northern Europe", "Southern Europe", "Central-Eastern Europe"])
R["school"] = both(lab("section"), ["High school", "Undergraduate", "Graduate"])
R["school_asia"] = both(lab("section"), ["High school", "Undergraduate"], keep=[info[t]["region"] == "Asia" for t in teams])
R["medal"] = both(lab("medal"), ["Gold", "Silver", "Bronze", "None"])
R["special"] = both(["Special prize" if info[t]["special"] else "No special prize" for t in teams], ["Special prize", "No special prize"])
R["nominated"] = both(["Nominated" if info[t]["nominated"] else "Not nominated" for t in teams], ["Nominated", "Not nominated"])
R["village"] = both(lab("village"), vil)

# medal ordered trend (None < Bronze < Silver < Gold): weighted slope of people share on rank, team shuffle
RANK = {"None": 0, "Bronze": 1, "Silver": 2, "Gold": 3}; rk = np.array([RANK[info[t]["medal"]] for t in teams], float)
def slope(r, n, k):
    w = n; x = r - (w * r).sum() / w.sum(); y = k / np.maximum(n[:, None], 1)
    return (w[:, None] * x[:, None] * y).sum(0) / (w * x * x).sum()
for lvl, K in (("main", KM), ("sub", KS)):
    o = slope(rk, NP, K); ge = np.zeros(K.shape[1])
    for _ in range(NPERM): ge += np.abs(slope(rk[rng.permutation(T)], NP, K)) >= np.abs(o) - 1e-12
    p = (ge + 1) / (NPERM + 1); R["medal"][lvl]["trend_p"] = p.round(4).tolist(); R["medal"][lvl]["trend_q"] = bh(p).round(4).tolist()
    R["medal"][lvl]["trend_slope_pp"] = (100 * o).round(2).tolist()

In [ ]:
# ---- theme prevalence + tone ------------------------------------------------------------------------------------
def prevalence(V, ids, name, parent, field):
    bs = np.empty((NBOOT, V.shape[1])); members = [np.where(tp == i)[0] for i in range(T)]
    for b in range(NBOOT):
        pick = rng.integers(0, T, T); rows = np.concatenate([members[i] for i in pick]); bs[b] = V[rows].mean(0)
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0); out = []
    for j, t in enumerate(ids):
        A = [r for r in M if int(r[field]) == t]; S_ = [r for r in A if r["sent_status"] == "scored"]
        c = collections.Counter(TONE[r["sent_label"]] for r in S_); qn = collections.Counter(r["question"] for r in A).most_common(1)[0]
        out.append(dict(id=t, name=name(t), short=SHORT.get(name(t), name(t)), parent=parent(t), people=int(V[:, j].sum()),
                        pct=round(V[:, j].mean(), 4), lo=round(lo[j], 4), hi=round(hi[j], 4),
                        teams=int(len(set(tp[V[:, j]]))), answers=len(A), tone=dict(n=len(S_), neg=c["neg"], mix=c["mix"], neu=c["neu"], pos=c["pos"]),
                        top_q=qn[0], top_q_share=round(qn[1] / len(A), 3)))
    return out
THM = dict(main=prevalence(VM, MID, lambda t: MAIN[t], lambda t: None, "main_id_wide"),
           sub=prevalence(VSA, SID_ALL, lambda t: SUB[t][0], lambda t: SUB[t][1], "sub_id_wide"))
tone_q = {}
for r in M:
    if r["sent_status"] == "scored":
        d = tone_q.setdefault(r["question"], dict(neg=0, mix=0, neu=0, pos=0)); d[TONE[r["sent_label"]]] += 1

# key differences in percentage points, people level, with team-bootstrap intervals
def diff(key, a, b, theme, lvl="main"):
    tb = R[key][lvl]; j = tb["themes"].index(theme); ga, gb = tb["groups"].index(a), tb["groups"].index(b)
    lab_ = {"school": lab("section"), "region": lab("region"), "medal": lab("medal")}[key]
    ia = [i for i in range(T) if lab_[i] == a]; ib = [i for i in range(T) if lab_[i] == b]; K = KM if lvl == "main" else KS; bs = []
    for _ in range(NBOOT):
        x = rng.choice(ia, len(ia)); y = rng.choice(ib, len(ib))
        bs.append(K[x, j].sum() / NP[x].sum() - K[y, j].sum() / NP[y].sum())
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return dict(key=key, theme=theme, a=a, b=b, pa=tb["share"][ga][j], pb=tb["share"][gb][j],
                pp=round(100 * (tb["share"][ga][j] - tb["share"][gb][j]), 1), lo=round(100 * lo, 1), hi=round(100 * hi, 1))
MONEY, INSP, CONF, BOND = MAIN[2], MAIN[1], MAIN[6], MAIN[5]
DIFFS = [diff("school", "Undergraduate", "High school", MONEY), diff("school", "Graduate", "High school", MONEY),
         diff("region", "North America", "Asia", MONEY), diff("school", "High school", "Undergraduate", BOND),
         diff("region", "Asia", "Europe", INSP), diff("medal", "Gold", "Silver", CONF)]

In [ ]:
# ---- forest plots (added 8 Nov, after his request): group A minus group B, percentage points, team bootstrap -------
# Rows: the 7 main themes + "positive tone" (% of scored answers that are positive). A and B are disjoint team sets.
# A dot is drawn filled when its 95% range excludes 0 AND the theme test of that comparison holds (q < 0.05; tone: p < 0.005 = 0.05 over the 10 comparisons).
def contrast(name, A_mask, B_mask, key):
    ia = [i for i in range(T) if A_mask[i]]; ib = [i for i in range(T) if B_mask[i]]
    # main themes, tone, then sub-themes (added on his request, same resamples, so the main numbers do not change)
    est = lambda x, y: np.concatenate([KM[x].sum(0) / NP[x].sum() - KM[y].sum(0) / NP[y].sum(),
                                       [TPOS[x].sum() / TN[x].sum() - TPOS[y].sum() / TN[y].sum()],
                                       KS[x].sum(0) / NP[x].sum() - KS[y].sum(0) / NP[y].sum()])
    e = est(ia, ib); bs = np.array([est(rng.choice(ia, len(ia)), rng.choice(ib, len(ib))) for _ in range(NBOOT)])
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0); tb = R[key]["main"]; ts = R[key]["sub"]; nm = len(MID)
    tone_sig = bool(tb["tone"]["p"] < .005 and (lo[nm] > 0 or hi[nm] < 0))   # tone: one test per comparison, 10 comparisons -> 0.05 / 10
    sig = [bool(tb["q"][j] < .05 and (lo[j] > 0 or hi[j] < 0)) for j in range(nm)] + [tone_sig]
    sub = list(range(nm + 1, nm + 1 + len(SID)))
    sig_sub = [bool(ts["q"][k] < .05 and (lo[i] > 0 or hi[i] < 0)) for k, i in enumerate(sub)] + [tone_sig]
    r1 = lambda v: np.round(100 * np.asarray(v), 1).tolist()
    sh = lambda x: np.concatenate([KM[x].sum(0) / NP[x].sum(), [TPOS[x].sum() / TN[x].sum()], KS[x].sum(0) / NP[x].sum()])
    sa, sb = sh(ia), sh(ib)
    base = dict(a_people=int(NP[ia].sum()), b_people=int(NP[ib].sum()), a_pct=r1(sa[:nm + 1]), b_pct=r1(sb[:nm + 1]),
                a_pct_sub=r1(list(sa[sub]) + [sa[nm]]), b_pct_sub=r1(list(sb[sub]) + [sb[nm]]))
    return dict(base, name=name, a_teams=len(ia), b_teams=len(ib), pp=r1(e[:nm + 1]), lo=r1(lo[:nm + 1]), hi=r1(hi[:nm + 1]), sig=sig,
                pp_sub=r1(list(e[sub]) + [e[nm]]), lo_sub=r1(list(lo[sub]) + [lo[nm]]), hi_sub=r1(list(hi[sub]) + [hi[nm]]), sig_sub=sig_sub)
L_ = {f: lab(f) for f in ("region", "section", "medal")}
is_ = lambda f, v: [x == v for x in L_[f]]
CON = {"rows": [MAIN[t] for t in MID] + ["Positive tone"], "rows_sub": [SUB[t][0] for t in SID] + ["Positive tone"]}
CON["school"] = [contrast("Undergraduate vs high school", is_("section", "Undergraduate"), is_("section", "High school"), "school"),
                 contrast("Graduate vs high school", is_("section", "Graduate"), is_("section", "High school"), "school")]
asia = is_("region", "Asia"); ug = is_("section", "Undergraduate")
CON["school_asia"] = [contrast("Undergraduate vs high school, Asian teams", [a and u for a, u in zip(asia, ug)], [a and h for a, h in zip(asia, is_("section", "High school"))], "school_asia")]
CON["region"] = [contrast("Asia vs Europe", is_("region", "Asia"), is_("region", "Europe"), "region"),
                 contrast("North America vs Europe", is_("region", "North America"), is_("region", "Europe"), "region")]
CON["region_undergrad"] = [contrast("Asia vs Europe, undergraduate teams", [a and u for a, u in zip(asia, ug)], [e and u for e, u in zip(is_("region", "Europe"), ug)], "region_undergrad"),
                           contrast("North America vs Europe, undergraduate teams", [a and u for a, u in zip(is_("region", "North America"), ug)], [e and u for e, u in zip(is_("region", "Europe"), ug)], "region_undergrad")]
CON["awards"] = [contrast("Gold medal vs other teams", is_("medal", "Gold"), [m != "Gold" for m in L_["medal"]], "medal"),
                 contrast("Special prize vs none", [info[t]["special"] for t in teams], [not info[t]["special"] for t in teams], "special"),
                 contrast("Nominated vs not", [info[t]["nominated"] for t in teams], [not info[t]["nominated"] for t in teams], "nominated")]
CON["country"] = [contrast(g + " vs all other teams", [x == g for x in lab_country], [x != g for x in lab_country], "country") for g in R["country"]["main"]["groups"] if g != "Other countries"]
CON["village"] = [contrast(g + " vs all other teams", [x == g for x in lab("village")], [x != g for x in lab("village")], "village") for g in vil]
CON["europe"] = [contrast(g + " vs rest of Europe", [x == g for x in lab_europe], [x not in ("-", g) for x in lab_europe], "europe") for g in R["europe"]["main"]["groups"]]

In [ ]:
# ---- where each theme comes from: answers per question x main theme (wide), for the flow figure ----------------
QS = ["Connection", "Team Work", "Beyond iGEM", "Conditions", "One Word"]
FLOW = dict(questions=QS, themes=MID + [-1], counts=[[sum(1 for r in M if r["question"] == q and int(r["main_id_wide"]) == t) for t in MID + [-1]] for q in QS])
assert sum(map(sum, FLOW["counts"])) == len(M)

In [ ]:
# ---- award ladder as one grouping (9 Nov, his request: nominated sits between no award and special prize) ------------------
# Computed last, so the random numbers of every table above stay the same.
lab_award = ["Special prize" if info[t]["special"] else "Nominated" if info[t]["nominated"] else "No award" for t in teams]
R["award_ladder"] = both(lab_award, ["No award", "Nominated", "Special prize"])

In [ ]:
# ---- France in detail (9 Nov, his request): counts only, no random numbers, so nothing above changes ------------------
fr_teams = [t for t in teams if info[t]["country"] == "FRA"]
fr_rows = [r for r in M if r["Team"] in fr_teams]
fr_people_all = sorted({r["Participant"] for r in fr_rows})
fr_size = sorted((len({r["Participant"] for r in fr_rows if r["Team"] == t}) for t in fr_teams), reverse=True)
FRANCE = dict(teams=len(fr_teams), people_answered=len(fr_people_all), people_themed=int(sum(NP[ti[t]] for t in fr_teams)), answers=len(fr_rows),
              largest_team_people=fr_size[0], median_team_people=fr_size[len(fr_size) // 2],
              school=dict(collections.Counter(info[t]["section"] for t in fr_teams)), medal=dict(collections.Counter(info[t]["medal"] for t in fr_teams)),
              award=dict(collections.Counter("Special prize" if info[t]["special"] else "Nominated" if info[t]["nominated"] else "No award" for t in fr_teams)),
              median_words=float(np.median([float(r["n_words"] or 0) for r in fr_rows])),
              median_words_rest=float(np.median([float(r["n_words"] or 0) for r in M if r["Team"] not in fr_teams])),
              no_theme_answers=sum(1 for r in fr_rows if r["main_id_wide"] == "-1"))
assert FRANCE["teams"] == 7

In [ ]:
# ---- context counts quoted in the page text (9 Nov final audit): counts only, no random numbers ----------------------------
CONTEXT = dict(
    median_words_by_region={g: float(np.median([float(r["n_words"] or 0) for r in M if info[r["Team"]]["region"] == g])) for g in ["Asia", "Europe", "North America", "Other regions"]},
    teams_region_by_school={sec: dict(collections.Counter(info[t]["region"] for t in teams if info[t]["section"] == sec)) for sec in ["High school", "Undergraduate", "Graduate"]},
    money_high_school_people=int(round(R["school"]["main"]["share"][0][MID.index(2)] * R["school"]["main"]["people"][0])))
print("Context counts:", CONTEXT)

# privacy: no group under 5 teams except medal Bronze (4), kept on his decision (8 Nov)
for key, v in R.items():
    for g, n in zip(v["main"]["groups"], v["main"]["teams"]):
        assert n >= 5 or (key == "medal" and g == "Bronze"), (key, g, n)
assert len(themed) == 328 and len(people_all) == 332 and T == 125
assert sum(THM["main"][j]["people"] for j in range(len(MID))) == int(VM.sum())

OUT = dict(meta=dict(assign="wide", base_people=len(themed), answered_people=len(people_all), teams=T, answers=len(M),
                     nperm=NPERM, nboot=NBOOT, seed=20261111, sub_min_people=15, n_sub_tested=len(SID), source="pooled_master.csv, Phase 2 run 2 Nov 2026"),
           themes=THM, tone_q=tone_q, groups=R, diffs=DIFFS, contrasts=CON, flow=FLOW, france=FRANCE, context=CONTEXT)
json.dump(OUT, open(OUT_DIR + "/igem-r4.json", "w", encoding="utf-8"), separators=(",", ":"), default=float)

In [ ]:
# ---- summary ----------------------------------------------------------------------------------------------------
L = [f"Round 4 (people level, wide view, base {len(themed)} people with a themed answer; {NPERM} team shuffles, {NBOOT} team resamples)."]
for key, v in R.items():
    for lvl in ("main", "sub"):
        tb = v[lvl]; hold = [f"{tb['themes'][j]} (q {tb['q'][j]:.3f}, LOO worst p {tb['loo'].get(tb['themes'][j], 'na')})" for j in range(len(tb["q"])) if tb["q"][j] < 0.05]
        cells = [f"{tb['groups'][g]} x {tb['themes'][j][:28]} {100*tb['share'][g][j]:.0f}% (r {tb['resid'][g][j]:+.1f})"
                 for g in range(len(tb["groups"])) for j in range(len(tb["themes"])) if tb["cell_q"][g][j] < 0.05 and tb["q"][j] < 0.05]
        L.append(f"[{key}/{lvl}] groups {list(zip(tb['groups'], tb['teams']))} | holds: {hold or 'none'} | rings: {cells or 'none'} | tone p {tb['tone']['p']}")
L.append("medal trend q (main): " + str(R["medal"]["main"]["trend_q"]))
for d in DIFFS: L.append(f"DIFF {d['theme'][:30]}: {d['a']} {100*d['pa']:.1f}% vs {d['b']} {100*d['pb']:.1f}% = {d['pp']:+} pp [{d['lo']}, {d['hi']}]")
for t in THM["main"]: L.append(f"THEME {t['short']}: {100*t['pct']:.1f}% [{100*t['lo']:.0f}-{100*t['hi']:.0f}] n={t['people']}")
open(OUT_DIR + "/nb4_part1_summary.txt", "w", encoding="utf-8").write("\n".join(L)); print("\n".join(L))

### Check against the website prototype

In [ ]:
# Check: part 1 must give the same numbers as the website prototype of 8 Nov 2026 (same data, same seed).
EXPECTED = {"teamwork_pct": 0.6951, "money_school_q": 0.0315, "money_country_q": 0.0087, "money_ug_vs_hs_pp": 23.8, "base_people": 328}
got = {"teamwork_pct": THM["main"][[t["id"] for t in THM["main"]].index(3)]["pct"],
       "money_school_q": R["school"]["main"]["q"][MID.index(2)],
       "money_country_q": R["country"]["main"]["q"][MID.index(2)],
       "money_ug_vs_hs_pp": CON["school"][0]["pp"][2],
       "base_people": len(themed)}
for k, v in EXPECTED.items():
    print(f"  {k:<20} expected {v}  got {got[k]}  ->", "same" if abs(got[k] - v) < 1e-9 else "DIFFERENT")
print("Part 1 reproduces the prototype:", all(abs(got[k] - v) < 1e-9 for k, v in EXPECTED.items()))

## Parts 2 to 5: new models

In [ ]:
# NB4 parts 2-5 (8 Nov 2026). Runs AFTER part 1 (the round-4 people-level analysis), whose variables it uses:
# M, teams, T, info, themed, tp, MID, SID, MAIN, SUB, VM, VS, NP, KM, KS, bh, OUT_DIR.
# Own random stream (seed 20261112), so part 1 numbers stay identical.
# Methods: proportional-odds (ordinal logistic) model, McCullagh (1980); proportionality checked with separate logits per cut,
# the idea of Brant (1990, Biometrics 46:1171-1178), here with a team bootstrap; GEE logistic clustered by team, Liang and Zeger (1986);
# Benjamini-Hochberg correction; smallest detectable difference with a team design effect (one-way ANOVA intra-class correlation).
import os, warnings, json, collections, math
import numpy as np, pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
warnings.filterwarnings("ignore")
rng2 = np.random.default_rng(20261112)
NPERM_ORD = int(os.environ.get("NB4_NPERM_ORD", "500"))     # team shuffles for the ordinal models (main themes)
NBOOT_PO = int(os.environ.get("NB4_NBOOT_PO", "300"))       # team bootstraps for the proportionality check

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Team table: the ladders and the controls
# ---------------------------------------------------------------------------------------------------------------
AWARD = np.array([2 if info[t]["special"] else 1 if info[t]["nominated"] else 0 for t in teams])        # none < nominated < special prize
MEDAL = np.array([{"None": 0, "Bronze": 1, "Silver": 2, "Gold": 3}[info[t]["medal"]] for t in teams])  # none < bronze < silver < gold
SCHOOL = np.array([{"High school": 0, "Undergraduate": 1, "Graduate": 2}[info[t]["section"]] for t in teams])
REGION = [info[t]["region"] for t in teams]
NRESP = np.array([len({r["Participant"] for r in M if r["Team"] == t}) for t in teams], float)          # respondents per team
def controls_team():
    X = pd.DataFrame({"Asia": [r == "Asia" for r in REGION], "North America": [r == "North America" for r in REGION],
                      "Other regions": [r == "Other regions" for r in REGION],
                      "Undergraduate": SCHOOL == 1, "Graduate": SCHOOL == 2, "log respondents": np.log(NRESP)}).astype(float)
    return X
XT = controls_team()
print("Team ladders (125 teams):")
print("  award ladder  none / nominated / special prize =", [int((AWARD == k).sum()) for k in range(3)])
print("  medal ladder  none / bronze / silver / gold     =", [int((MEDAL == k).sum()) for k in range(4)])
assert (AWARD == 0).sum() + (AWARD == 1).sum() + (AWARD == 2).sum() == T == 125

def fit_ordinal(y, X, groups=None):
    cov = dict(cov_type="cluster", cov_kwds={"groups": groups}) if groups is not None else {}
    m = OrderedModel(pd.Series(pd.Categorical(y, ordered=True)), X, distr="logit")
    return m.fit(method="bfgs", disp=False, maxiter=3000, **cov)

def share_matrix(level):
    K = KM if level == "main" else KS
    return K / NP[:, None]

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# PART 2 (A): does talking more about a theme go with climbing the award / medal ladder?  (team level)
# ---------------------------------------------------------------------------------------------------------------
def ordinal_ladder(outcome_name, Y, level):
    S = share_matrix(level); names = [MAIN[t] for t in MID] if level == "main" else [SUB[t][0] for t in SID]
    rows = []
    for j, nm in enumerate(names):
        X = XT.copy(); X.insert(0, "theme share (+10 points)", S[:, j] * 10)
        try:
            r = fit_ordinal(Y, X); b, se = r.params.iloc[0], r.bse.iloc[0]; ok = bool(np.isfinite(se) and se > 0)
        except Exception:
            b, se, ok = np.nan, np.nan, False
        row = dict(outcome=outcome_name, level=level, theme=nm, OR=float(np.exp(b)) if ok else None,
                   lo=float(np.exp(b - 1.96 * se)) if ok else None, hi=float(np.exp(b + 1.96 * se)) if ok else None,
                   p=float(2 * (1 - __import__("scipy").stats.norm.cdf(abs(b / se)))) if ok else None, fitted=ok)
        # plain reading: chance of reaching the next step, low (25th percentile) vs high (75th percentile) theme share
        if ok:
            lo_s, hi_s = np.percentile(S[:, j], [25, 75]); base = X.mean().values.copy()
            thr = r.params.values[X.shape[1]:]; cut = thr[0]                       # first threshold: none vs at least step 1
            def p_step1(s): xb = base.copy(); xb[0] = s * 10; return 1 - 1 / (1 + np.exp(-(cut - xb @ r.params.values[:X.shape[1]])))
            row.update(share_low=round(float(lo_s), 3), share_high=round(float(hi_s), 3),
                       p_step1_low=round(float(p_step1(lo_s)), 3), p_step1_high=round(float(p_step1(hi_s)), 3))
        rows.append(row)
    ps = [r["p"] for r in rows if r["p"] is not None]; qs = bh(ps) if ps else []; k = 0
    for r in rows:
        if r["p"] is not None: r["q"] = float(qs[k]); k += 1
        else: r["q"] = None
    return rows

def perm_ordinal(Y, level, rows):
    """Team shuffle: the ladder is shuffled across teams, the model refitted; p = share of shuffles with |z| at least the observed.
    Run only for themes close to holding (q < 0.10): the shuffle checks a possible finding; far-from-significant themes do not need it."""
    S = share_matrix(level)
    for j, r in enumerate(rows):
        if not r["fitted"] or r["q"] is None or r["q"] >= 0.10: r["p_shuffle"] = None; continue
        X = XT.copy(); X.insert(0, "s", S[:, j] * 10)
        z0 = abs(math.log(r["OR"]) / ((math.log(r["hi"]) - math.log(r["lo"])) / 3.92)); ge = 0; n = 0
        for _ in range(NPERM_ORD):
            try:
                rr = fit_ordinal(Y[rng2.permutation(T)], X); z = abs(rr.params.iloc[0] / rr.bse.iloc[0])
                ge += z >= z0 - 1e-9; n += 1
            except Exception:
                pass
        r["p_shuffle"] = round((ge + 1) / (n + 1), 4)

def po_check(Y, level, rows, ncut):
    """Proportional-odds check: separate logits for each cut (y >= k); team bootstrap of the difference between the
    theme coefficients of the highest and the lowest usable cut. Only cuts with at least 10 teams on each side are used."""
    S = share_matrix(level)
    cuts = [k for k in range(1, ncut) if (Y >= k).sum() >= 10 and (Y < k).sum() >= 10]
    for j, r in enumerate(rows):
        r["po_cuts"] = cuts; r["po_flag"] = None
        if not r["fitted"] or len(cuts) < 2: continue
        X = sm.add_constant(pd.concat([pd.Series(S[:, j] * 10, name="s"), XT], axis=1))
        def coefs(idx):
            out = []
            for k in (cuts[0], cuts[-1]):
                try: out.append(sm.Logit((Y[idx] >= k).astype(float), X.iloc[idx]).fit(disp=False, maxiter=200).params["s"])
                except Exception: out.append(np.nan)
            return out
        c = coefs(np.arange(T)); d = []
        for _ in range(NBOOT_PO):
            b = rng2.integers(0, T, T); cb = coefs(b)
            if np.all(np.isfinite(cb)): d.append(cb[1] - cb[0])
        if len(d) > 50:
            lo, hi = np.percentile(d, [2.5, 97.5])
            r.update(OR_cut_low=round(float(np.exp(c[0])), 3), OR_cut_high=round(float(np.exp(c[1])), 3), po_flag=bool(lo > 0 or hi < 0))

ORD = []
for oname, Y, ncut in (("award ladder", AWARD, 3), ("medal ladder", MEDAL, 4)):
    for level in ("main", "sub"):
        rows = ordinal_ladder(oname, Y, level)
        if level == "main": perm_ordinal(Y, level, rows); po_check(Y, level, rows, ncut)
        ORD += rows
# sensitivity for the medal ladder: gold vs the other teams, team-level logistic model with the same controls
GOLD = (MEDAL == 3).astype(float); S_main = share_matrix("main")
for j, r in enumerate([r for r in ORD if r["outcome"] == "medal ladder" and r["level"] == "main"]):
    X = sm.add_constant(pd.concat([pd.Series(S_main[:, j] * 10, name="s"), XT], axis=1))
    try:
        g = sm.Logit(GOLD, X).fit(disp=False, maxiter=200); b, se = g.params["s"], g.bse["s"]
        r.update(gold_OR=round(float(np.exp(b)), 3), gold_lo=round(float(np.exp(b - 1.96 * se)), 3), gold_hi=round(float(np.exp(b + 1.96 * se)), 3),
                 gold_p=round(float(g.pvalues["s"]), 4))
    except Exception:
        r.update(gold_OR=None)
gp = [r for r in ORD if r["outcome"] == "medal ladder" and r["level"] == "main" and r.get("gold_OR")]
for r, q in zip(gp, bh([r["gold_p"] for r in gp])): r["gold_q"] = round(float(q), 4)

print("\nPART 2 - award and medal ladders (odds ratio per +10 points of a team's people who raised the theme; 1 = no link)")
for r in ORD:
    if r["level"] != "main": continue
    if not r["fitted"]: print(f"  {r['outcome']:<13} {r['theme'][:42]:<42} model did not fit"); continue
    flag = " (ladder assumption doubtful)" if r.get("po_flag") else ""
    extra = f" | gold vs rest OR {r['gold_OR']:.2f} [{r['gold_lo']:.2f}, {r['gold_hi']:.2f}] q {r['gold_q']:.2f}" if r.get("gold_OR") else ""
    print(f"  {r['outcome']:<13} {r['theme'][:42]:<42} OR {r['OR']:.2f} [{r['lo']:.2f}, {r['hi']:.2f}]  q {r['q']:.2f}  shuffle p {r['p_shuffle'] if r['p_shuffle'] is not None else 'not needed (q >= 0.10)'}{flag}{extra}")
for oname in ("award ladder", "medal ladder"):
    for level in ("main", "sub"):
        sub = [r for r in ORD if r["outcome"] == oname and r["level"] == level and r["q"] is not None]
        print(f"  -> {oname}, {level}: {sum(r['q'] < .05 for r in sub)} of {len(sub)} themes hold after correction; {sum(not r['fitted'] for r in ORD if r['outcome']==oname and r['level']==level)} did not fit")

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# PART 3 (C): all factors in one model, on people (GEE logistic, people clustered by team)
# ---------------------------------------------------------------------------------------------------------------
words = collections.defaultdict(list)
for r in M: words[r["Participant"]].append(float(r["n_words"] or 0))
FACT = ["Asia vs Europe", "North America vs Europe", "Other regions vs Europe", "Undergraduate vs high school", "Graduate vs high school",
        "medal ladder (+1 step)", "award ladder (+1 step)", "team respondents (log)", "answer length (log words)"]
def people_X():
    rows = []
    for k, p in enumerate(themed):
        i = tp[k]; reg = REGION[i]
        rows.append([reg == "Asia", reg == "North America", reg == "Other regions", SCHOOL[i] == 1, SCHOOL[i] == 2,
                     MEDAL[i], AWARD[i], math.log(NRESP[i]), math.log(1 + np.mean(words[p]))])
    return pd.DataFrame(rows, columns=FACT).astype(float)
XP = sm.add_constant(people_X()); GROUPS = tp
def gee_theme(y):
    m = sm.GEE(y.astype(float), XP, groups=GROUPS, family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable())
    return m.fit(maxiter=200)
COMB = []
for level, V, names in (("main", VM, [MAIN[t] for t in MID]), ("sub", VS, [SUB[t][0] for t in SID])):
    res = []
    DUMMY = FACT[:5]
    for j, nm in enumerate(names):
        y = V[:, j].astype(float)
        # separation guard: a 0/1 factor whose group has nobody (or everybody) raising the theme cannot be estimated
        empty = [f for f in DUMMY if XP[f].sum() > 0 and (y[XP[f].values == 1].sum() in (0, XP[f].sum()))]
        Xj = XP.drop(columns=empty)
        try:
            m = sm.GEE(y, Xj, groups=GROUPS, family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable()).fit(maxiter=200)
            est = {f: (m.params[f], m.bse[f]) for f in FACT if f not in empty}
            ok = bool(all(np.isfinite(se) and 0 < se < 20 and abs(b) < 7 for b, se in est.values()))
        except Exception:
            est, ok = {}, False
        res.append((nm, ok, est, empty))
    for f in FACT:
        cand = [(nm, est[f]) for nm, ok, est, empty in res if ok and f in est]
        ps = [2 * (1 - __import__("scipy").stats.norm.cdf(abs(b / se))) for nm, (b, se) in cand]; qs = bh(ps) if ps else []
        pq = {nm: (p, q) for (nm, _), p, q in zip(cand, ps, qs)}
        for nm, ok, est, empty in res:
            row = dict(level=level, theme=nm, factor=f, fitted=bool(ok and f in est), note="no person in this group raised it" if f in empty else ("" if ok else "model did not fit"))
            if row["fitted"]:
                b, se = est[f]; p, q = pq[nm]
                row.update(OR=float(np.exp(b)), lo=float(np.exp(b - 1.96 * se)), hi=float(np.exp(b + 1.96 * se)), p=float(p), q=float(q))
            COMB.append(row)
print("\nPART 3 - all factors in one model, on people (odds ratio; 1 = no link). Results that hold after correction:")
hold = [r for r in COMB if r["fitted"] and r["q"] < .05]
for r in hold: print(f"  [{r['level']}] {r['theme'][:40]:<40} {r['factor']:<30} OR {r['OR']:.2f} [{r['lo']:.2f}, {r['hi']:.2f}] q {r['q']:.3f}")
if not hold: print("  none")
for level in ("main", "sub"):
    print(f"  -> {level}: {len({r['theme'] for r in COMB if r['level']==level and r['note']=='model did not fit'})} themes did not fit; "
          f"{sum(1 for r in COMB if r['level']==level and r['note'].startswith('no person'))} theme-factor pairs not estimable (nobody in that group raised the theme)")

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# PART 4 (D): tone as a ladder (negative < neutral or mixed < positive), answer level, clustered by team
# ---------------------------------------------------------------------------------------------------------------
TONE3 = {"strongly_negative": 0, "negative": 0, "mixed": 1, "neutral": 1, "positive": 2, "strongly_positive": 2}
A = [r for r in M if r["sent_status"] == "scored"]; ti_ = {t: i for i, t in enumerate(teams)}
yA = np.array([TONE3[r["sent_label"]] for r in A]); gA = np.array([ti_[r["Team"]] for r in A])
Qs = ["Team Work", "Beyond iGEM", "Conditions", "One Word"]                     # reference: Connection
def xa(cols):
    D = {}
    if "question" in cols:
        for q in Qs: D["question: " + q] = [r["question"] == q for r in A]
    if "groups" in cols:
        D.update({"Asia vs Europe": [REGION[ti_[r["Team"]]] == "Asia" for r in A], "North America vs Europe": [REGION[ti_[r["Team"]]] == "North America" for r in A],
                  "Other regions vs Europe": [REGION[ti_[r["Team"]]] == "Other regions" for r in A],
                  "Undergraduate vs high school": [SCHOOL[ti_[r["Team"]]] == 1 for r in A], "Graduate vs high school": [SCHOOL[ti_[r["Team"]]] == 2 for r in A],
                  "medal ladder (+1 step)": [MEDAL[ti_[r["Team"]]] for r in A], "award ladder (+1 step)": [AWARD[ti_[r["Team"]]] for r in A]})
    if "length" in cols: D["answer length (log words)"] = [math.log(1 + float(r["n_words"] or 0)) for r in A]
    return pd.DataFrame(D).astype(float)
TONE_FITS = {}
for name, cols in (("question only", ["question"]), ("groups only", ["groups"]), ("all", ["question", "groups", "length"])):
    TONE_FITS[name] = fit_ordinal(yA, xa(cols), groups=gA)
rA = TONE_FITS["all"]; TONE = []
gf = [c for c in xa(["question", "groups", "length"]).columns]
pt = [float(rA.pvalues[c]) for c in gf]; qt = bh(pt)
for c, p, q in zip(gf, pt, qt):
    TONE.append(dict(factor=c, OR=float(np.exp(rA.params[c])), lo=float(np.exp(rA.params[c] - 1.96 * rA.bse[c])), hi=float(np.exp(rA.params[c] + 1.96 * rA.bse[c])), p=p, q=float(q)))
R2 = {k: round(float(v.prsquared), 4) for k, v in TONE_FITS.items()}
print("\nPART 4 - tone ladder (odds ratio of a more positive answer; 1 = no link; reference question: 'When did you feel most connected?')")
for r in TONE: print(f"  {r['factor']:<32} OR {r['OR']:.2f} [{r['lo']:.2f}, {r['hi']:.2f}]  q {r['q']:.3f}")
print(f"  How much each block explains (McFadden pseudo R2): question only {R2['question only']}, groups only {R2['groups only']}, all {R2['all']}")

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# PART 5 (F): smallest difference each two-group comparison could detect (80% power, 5% two-sided), main themes
# ---------------------------------------------------------------------------------------------------------------
def icc(y):
    """one-way ANOVA intra-class correlation of a 0/1 outcome, people within teams"""
    k = T; n = NP; N = n.sum(); g = np.array([y[tp == i].sum() for i in range(T)]); mean_i = g / np.maximum(n, 1); mu = y.mean()
    msb = (n * (mean_i - mu) ** 2).sum() / (k - 1); msw = sum(((y[tp == i] - mean_i[i]) ** 2).sum() for i in range(T)) / (N - k)
    n0 = (N - (n ** 2).sum() / N) / (k - 1); return max(0.0, (msb - msw) / (msb + (n0 - 1) * msw)), n0
PAIRS = {"Undergraduate vs high school": (SCHOOL == 1, SCHOOL == 0), "Asia vs Europe": (np.array(REGION) == "Asia", np.array(REGION) == "Europe"),
         "gold medal vs other teams": (MEDAL == 3, MEDAL < 3), "special prize vs none": (AWARD == 2, AWARD == 0)}
POWER = []
for j, t in enumerate(MID):
    y = VM[:, j].astype(float); rho, n0 = icc(y); p0 = y.mean(); deff = 1 + (n0 - 1) * rho
    for pname, (ma, mb) in PAIRS.items():
        nA = NP[ma].sum(); nB = NP[mb].sum()
        mde = (1.959964 + 0.841621) * math.sqrt(p0 * (1 - p0) * (1 / nA + 1 / nB) * deff)
        POWER.append(dict(theme=MAIN[t], comparison=pname, people_a=int(nA), people_b=int(nB), icc=round(rho, 3), design_effect=round(deff, 2), mde_points=round(100 * mde, 1)))
print("\nPART 5 - smallest difference each comparison could detect (percentage points; 80% power, teams taken into account)")
for pname in PAIRS:
    vals = [r["mde_points"] for r in POWER if r["comparison"] == pname]
    print(f"  {pname:<30} from {min(vals):.0f} to {max(vals):.0f} points, depending on the theme")

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# PART 2b (B, 9 Nov, his request): does the share raising a theme go up step by step on the ladder?
# Regression of each person's 0/1 (raised the theme) on the team's ladder step (award 0/1/2, medal 0/1/2/3):
# the slope = percentage points per step. 95% range: 2,000 resamples of whole teams. Test: 2,000 shuffles of the
# ladder between teams; Benjamini-Hochberg over the themes. Own random stream, so nothing above changes.
# ---------------------------------------------------------------------------------------------------------------
rng3 = np.random.default_rng(20261113)
def slopes(x, V):
    xc = x - x.mean(); return (xc[:, None] * V).sum(0) / (xc ** 2).sum()
members_t = [np.where(tp == i)[0] for i in range(T)]
TREND = []
for oname, Ylad in (("award ladder", AWARD), ("medal ladder", MEDAL)):
    xp = Ylad[tp].astype(float)
    for level, V, names in (("main", VM.astype(float), [MAIN[t] for t in MID]), ("sub", VS.astype(float), [SUB[t][0] for t in SID])):
        b0 = slopes(xp, V)
        bs = np.empty((2000, V.shape[1]))
        for k in range(2000):
            pick = rng3.integers(0, T, T); rows_ = np.concatenate([members_t[i] for i in pick]); bs[k] = slopes(xp[rows_], V[rows_])
        lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
        ge = np.zeros(V.shape[1])
        for k in range(2000):
            ge += np.abs(slopes(Ylad[rng3.permutation(T)][tp].astype(float), V)) >= np.abs(b0) - 1e-12
        p = (ge + 1) / 2001; q = bh(p)
        for j, nm in enumerate(names):
            TREND.append(dict(outcome=oname, level=level, theme=nm, pp_per_step=round(float(100 * b0[j]), 2), lo=round(float(100 * lo[j]), 2), hi=round(float(100 * hi[j]), 2),
                              p=round(float(p[j]), 4), q=round(float(q[j]), 4)))
print("\nPART 2b - change in the share raising a theme per step up the ladder (percentage points; team shuffle, corrected)")
for r in TREND:
    if r["level"] == "main":
        print(f"  {r['outcome']:<13} {r['theme'][:42]:<42} {r['pp_per_step']:+.1f} points per step [{r['lo']:+.1f}, {r['hi']:+.1f}]  q {r['q']:.3f}")
for oname in ("award ladder", "medal ladder"):
    for level in ("main", "sub"):
        sub_ = [r for r in TREND if r["outcome"] == oname and r["level"] == level]
        print(f"  -> {oname}, {level}: {sum(r['q'] < .05 for r in sub_)} of {len(sub_)} themes hold after correction")

In [ ]:
# ---------------------------------------------------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------------------------------------------------
pd.DataFrame(ORD).to_csv(OUT_DIR + "/nb4_ordinal_ladders.csv", index=False)
pd.DataFrame(COMB).to_csv(OUT_DIR + "/nb4_combined_model_people.csv", index=False)
pd.DataFrame(TONE).to_csv(OUT_DIR + "/nb4_tone_ladder.csv", index=False)
pd.DataFrame(POWER).to_csv(OUT_DIR + "/nb4_power.csv", index=False)
pd.DataFrame(TREND).to_csv(OUT_DIR + "/nb4_ladder_trend.csv", index=False)
R5 = dict(meta=dict(seed=20261112, nperm_ord=NPERM_ORD, nboot_po=NBOOT_PO, teams=T, people=len(themed), scored_answers=len(A),
                    award_ladder=[int((AWARD == k).sum()) for k in range(3)], medal_ladder=[int((MEDAL == k).sum()) for k in range(4)]),
          ordinal=ORD, combined=COMB, tone=TONE, tone_r2=R2, power=POWER, trend=TREND)
json.dump(R5, open(OUT_DIR + "/igem-r5.json", "w", encoding="utf-8"), separators=(",", ":"), default=float)
print("\nSaved: nb4_ordinal_ladders.csv, nb4_combined_model_people.csv, nb4_tone_ladder.csv, nb4_power.csv, igem-r5.json in", OUT_DIR)

## Download the results

In [ ]:
# Pack everything into one zip and download it. The zip holds the outputs.
import shutil
stamp = datetime.date.today().isoformat()
z = shutil.make_archive(f"/content/NB4_outputs_{stamp}" if IN_COLAB else OUT_DIR + f"_zip_{stamp}", "zip", OUT_DIR)
print("Zip:", z); print("Files:", sorted(os.listdir(OUT_DIR)))
if IN_COLAB:
    from google.colab import files; files.download(z)